<a href="https://colab.research.google.com/github/Alexander12Po/APKDE-PLAGA/blob/main/colab_build_apk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compilar Agrowillay (APK) en Google Colab

Este notebook clona tu repositorio de GitHub, instala Buildozer y todas las
dependencias del sistema, compila el APK y lo deja listo para descargar.

**Antes de empezar:** sube tu proyecto (main.py, buildozer.spec, assets/) a
un repositorio de GitHub. No subas ningun archivo `config.json` ni claves
de API.

Ejecuta las celdas en orden, de arriba hacia abajo.

In [25]:
# 1) Clonar tu repositorio de GitHub
REPO_URL = "https://github.com/Alexander12Po/APKDE-PLAGA.git"
PROJECT_DIR = "APKDE-PLAGA"

# Empezar siempre desde /content evita APKDE-PLAGA/APKDE-PLAGA.
%cd /content
!rm -rf APKDE-PLAGA
!git clone https://github.com/Alexander12Po/APKDE-PLAGA.git APKDE-PLAGA
%cd /content/APKDE-PLAGA

print("===== COMMIT DESCARGADO =====")
!git log -1 --oneline

print("===== CONFIGURACIÓN ANDROID =====")
!grep -E '^(android\.api|android\.minapi|android\.ndk|android\.sdk|p4a\.branch)' buildozer.spec || true

print("===== ARCHIVOS DEL PROYECTO =====")
!ls -la


fatal: destination path 'APKDE-PLAGA' already exists and is not an empty directory.
/content/APKDE-PLAGA/APKDE-PLAGA
total 820
drwxr-xr-x 4 root root   4096 Aug 27 15:47 .
drwxr-xr-x 6 root root   4096 Aug 27 15:47 ..
drwxr-xr-x 3 root root   4096 Aug 27 15:47 APKDE-PLAGA
-rw-r--r-- 1 root root   3023 Aug 27 15:47 buildozer.spec
-rw-r--r-- 1 root root   5535 Aug 27 15:47 colab_build_apk.ipynb
drwxr-xr-x 8 root root   4096 Aug 27 15:47 .git
-rw-r--r-- 1 root root 311735 Aug 27 15:47 icon.png
-rw-r--r-- 1 root root  25331 Aug 27 15:47 main.py
-rw-r--r-- 1 root root 462283 Aug 27 15:47 presplash.png
-rw-r--r-- 1 root root   3036 Aug 27 15:47 README.md


In [26]:
!rm -rf .buildozer ~/.buildozer

In [27]:
!cat buildozer.spec

[app]

# ---------------------------------------------------------------------------
# Identidad de la app
# ---------------------------------------------------------------------------
title = Agrowillay
package.name = agrowillay
package.domain = org.agrowillay

# Carpeta fuente (donde esta main.py)
source.dir = .
source.include_exts = py,png,jpg,jpeg,kv,atlas,ttf,json

version = 1.0.0

# ---------------------------------------------------------------------------
# Requerimientos de Python
# ---------------------------------------------------------------------------
# NOTA: no incluimos "google-generativeai" porque main.py llama a la API de
# Gemini directamente por REST (con "requests"), lo cual es mucho mas liviano
# y evita dependencias problematicas de compilar para Android (grpc, protobuf).
requirements = python3,kivy==2.3.0,kivymd==1.2.0,requests,pillow,plyer,certifi,urllib3,charset-normalizer,idna

# ---------------------------------------------------------------------------
# R

## 2) Instalar dependencias del sistema

Buildozer necesita Java, herramientas de compilacion de Android, y varias librerias de sistema Linux.

In [28]:
%%bash
set -e

apt-get update -qq
apt-get install -y -qq \
    openjdk-17-jdk \
    autoconf \
    automake \
    libtool \
    pkg-config \
    zlib1g-dev \
    libncurses5-dev \
    libncursesw5-dev \
    libtinfo5 \
    cmake \
    libffi-dev \
    libssl-dev \
    unzip \
    zip \
    build-essential \
    ccache \
    git

echo "Dependencias del sistema instaladas."


Dependencias del sistema instaladas.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [29]:
# 3) Instalar Buildozer y Cython
!pip install --upgrade pip
!pip install buildozer cython
!buildozer version

Buildozer is running as root!
This is not recommended, and may lead to problems later.
Are you sure you want to continue [y/n]? y
Buildozer 1.6.0


## 4) Configurar variables de entorno de Java

Colab a veces trae varias versiones de Java; forzamos la que necesita el
Android SDK/NDK.

In [30]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version

openjdk version "17.0.20" 2026-07-21
OpenJDK Runtime Environment (build 17.0.20+8-1-22.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20+8-1-22.04-Ubuntu, mixed mode, sharing)


## 5) Compilar el APK

Esto puede tardar entre 15 y 40 minutos la primera vez (descarga el
Android SDK/NDK completos). Las siguientes compilaciones son mas rapidas
porque Buildozer cachea el SDK/NDK en `.buildozer/`.

**Importante:** si la sesion de Colab se desconecta, vuelve a ejecutar las
celdas desde el paso 1 (o desde el paso 4 si `.buildozer/` sigue en disco).

In [31]:
# 5) Preparar Android SDK y compilar el APK
%cd /content/{PROJECT_DIR}

import os, glob, subprocess

ANDROIDSDK = os.path.expanduser("~/.buildozer/android/platform/android-sdk")
os.environ["ANDROIDSDK"] = ANDROIDSDK
os.environ["ANDROID_HOME"] = ANDROIDSDK
os.environ["ANDROIDAPI"] = "36"
os.environ["ANDROIDMINAPI"] = "24"
os.makedirs(ANDROIDSDK, exist_ok=True)

def find_sdkmanager():
    paths = glob.glob(ANDROIDSDK + "/**/sdkmanager", recursive=True)
    return paths[0] if paths else None

sdkmanager = find_sdkmanager()

if not sdkmanager:
    print("Instalando Android command-line tools...")
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip -O /tmp/cmdline-tools.zip
    !rm -rf /tmp/cmdline-tools
    !mkdir -p /tmp/cmdline-tools
    !unzip -q /tmp/cmdline-tools.zip -d /tmp/cmdline-tools
    !rm -rf "{ANDROIDSDK}/cmdline-tools"
    !mkdir -p "{ANDROIDSDK}/cmdline-tools/latest"
    !cp -r /tmp/cmdline-tools/cmdline-tools/* "{ANDROIDSDK}/cmdline-tools/latest/"
    sdkmanager = ANDROIDSDK + "/cmdline-tools/latest/bin/sdkmanager"

print("sdkmanager:", sdkmanager)

subprocess.run([sdkmanager, "--sdk_root=" + ANDROIDSDK, "--licenses"], input=("y\n" * 50).encode(), check=False)
subprocess.run([
    sdkmanager, "--sdk_root=" + ANDROIDSDK,
    "platform-tools",
    "platforms;android-36",
    "build-tools;36.0.0"
], check=False)

print("===== COMPROBACIÓN SDK =====")
!ls -d "{ANDROIDSDK}/platforms/android-36"
!ls -d "{ANDROIDSDK}/build-tools/36.0.0"

# Borrar únicamente la caché del proyecto para evitar usar una compilación vieja.
!rm -rf .buildozer

print("===== BUILD APK =====")
env = os.environ.copy()
env["ANDROIDSDK"] = ANDROIDSDK
env["ANDROID_HOME"] = ANDROIDSDK
env["ANDROIDAPI"] = "36"
env["ANDROIDMINAPI"] = "24"
subprocess.run(["bash", "-lc", "yes | buildozer -v android debug"], env=env, check=True)


/content/APKDE-PLAGA
Buildozer is running as root!
This is not recommended, and may lead to problems later.
Are you sure you want to continue [y/n]? # WARNING: Config token app android.sdk is deprecated and ignored, but you set value 34
# Ensure build layout
# Create directory /root/.buildozer
# Create directory /root/.buildozer/cache
# Create directory /root/.buildozer/android/platform/android/platform
# Check configuration tokens
# Preparing build
# Check requirements for android
# Search for Git (git)
#  -> found at /usr/bin/git
# Search for Cython (cython)
#  -> found at /usr/local/bin/cython
# Search for Java compiler (javac)
#  -> found at /usr/lib/jvm/java-17-openjdk-amd64/bin/javac
# Search for Java keytool (keytool)
#  -> found at /usr/lib/jvm/java-17-openjdk-amd64/bin/keytool
# Install platform
# Run 'git config --get remote.origin.url' ...
# Cwd /content/APKDE-PLAGA/.buildozer/android/platform/python-for-android
https://github.com/kivy/python-for-android.git
# Run 'git branc

## 6) Ubicar y descargar el APK compilado

El archivo queda dentro de la carpeta `bin/`.

In [32]:
import glob, os
apk_files = sorted(glob.glob("/content/{}/bin/*.apk".format(PROJECT_DIR)))
print("APK generado:", apk_files)
if apk_files:
    for apk in apk_files:
        print(f"{os.path.basename(apk)} — {os.path.getsize(apk) / (1024*1024):.2f} MB")


APK generado: []


In [33]:
from google.colab import files

if apk_files:
    files.download(apk_files[-1])
else:
    print("No se encontró ningún APK. Revisa el log de compilación.")


No se encontro ningun APK. Revisa el log de buildozer arriba para ver el error de compilacion.


## Notas y solución de problemas

- **"license not accepted"**: el notebook acepta las licencias del SDK automáticamente.
- **API 34 no disponible**: este notebook fuerza `ANDROIDAPI=36` y prepara `platforms;android-36`.
- **Carpeta duplicada `APKDE-PLAGA/APKDE-PLAGA`**: el paso 1 siempre comienza desde `/content` y elimina la copia anterior antes de clonar.
- **Cambios en el código**: si editas `main.py`, `buildozer.spec` u otro archivo en GitHub, vuelve a ejecutar el notebook desde el paso 1 para clonar la versión actual del repositorio.
- **Sesión desconectada**: vuelve a ejecutar las celdas desde el paso 1. La sesión de Colab es temporal.
- **APK**: el archivo compilado queda dentro de `bin/` y el último paso lo descarga automáticamente.
- **Versión release**: para una compilación de distribución, reemplaza `android debug` por `android release` y firma el resultado con tu keystore.
